In [1]:
# ── Cài Google Chrome (thay thế Chromium snap) ─────────────────
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q

# ── Cài ChromeDriver khớp version ──────────────────────────────
!pip install selenium webdriver-manager -q

# ── Kiểm tra ───────────────────────────────────────────────────
import subprocess
print(subprocess.getoutput("google-chrome --version"))

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 3 not upgraded.
Need to get 11.2 MB/141 MB of archives.
After this operation, 478 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-0 amd64 2.36.0-3build1 [51.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatspi2.0-0 a

In [ ]:
# ── Giải phóng RAM ───────────────────────────────────────────────
import gc, os, subprocess

# Kill chrome process cũ nếu còn
subprocess.run(["pkill", "-f", "chrome"], capture_output=True)
subprocess.run(["pkill", "-f", "chromedriver"], capture_output=True)

# Tăng /dev/shm (shared memory) cho Chrome
subprocess.run(["mount", "-o", "remount,size=2G", "/dev/shm"],
               capture_output=True)

gc.collect()

# Kiểm tra RAM còn lại
mem = subprocess.getoutput("free -h")
shm = subprocess.getoutput("df -h /dev/shm")
print("RAM:\n", mem)
print("\n/dev/shm:\n", shm)

RAM:
                total        used        free      shared  buff/cache   available
Mem:            12Gi       923Mi       7.9Gi       2.0Mi       3.8Gi        11Gi
Swap:             0B          0B          0B

/dev/shm:
 Filesystem      Size  Used Avail Use% Mounted on
shm             2.0G     0  2.0G   0% /dev/shm


In [ ]:
"""
WhoScored Premier League 2025-2026 - MOTM Crawler
==================================================
Output columns (theo field_analysis.md):

match_id, match_date, season,
home_team, away_team, home_score, away_score,
player_id, name, team, is_home, position, age,
is_first_eleven, is_man_of_match, minutes_played,
rating, goals, assists,
shots_total, shots_on_target, key_passes,
passes_completed, passes_total, pass_accuracy,
tackles, interceptions, clearances,
aerial_won, aerial_lost,
dribbles_won, dribbles_attempted,
fouls_committed, saves

Loại bỏ: big_chances_created/missed, yellow_cards, red_cards,
          competition, shirt_no, fouls_drawn

Cài đặt Colab:
    !wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
    !apt-get install -y ./google-chrome-stable_current_amd64.deb -q
    !pip install selenium webdriver-manager pandas -q
"""

import re
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
FIXTURES_URL = (
    "https://www.whoscored.com/regions/252/tournaments/2/seasons/10743"
    "/stages/24533/fixtures"
)

TARGET_MONTHS = [
    "May 2026", "Apr 2026", "Mar 2026", "Feb 2026", "Jan 2026",
    "Dec 2025", "Nov 2025", "Oct 2025", "Sep 2025", "Aug 2025"
]
MONTH_ORDER = {m: i for i, m in enumerate(reversed(TARGET_MONTHS))}

SEL_PREV_BTN    = "button.Calendar-module_dayChangeBtn__sEvC8"
SEL_MONTH_LABEL = "button.Calendar-module_toggleCalendar__SI3Dq"
SEL_SCORE       = "a.Match-module_score__5Ghhj"
SEL_TEAM_NAME   = "a.Match-module_teamNameText__Dqv-G"
SEL_STATS_BTN   = "a.Match-module_statsBtn__O2q4H"

# Tab → WhoScored header → field name
TAB_FIELD_MAP = {
    "Summary": {
        "Shots":      "shots_total",       # ← chỉ lấy ở Summary
        "ShotsOT":    "shots_on_target",
        "KeyPasses":  "key_passes",
        "PA%":        "pass_accuracy",
        "AerialsWon": "aerial_won",
        "Rating":     "rating",
    },
    "Offensive": {
        # Bỏ Shots/ShotsOT ở đây vì đã có từ Summary
        "Dribbles": "dribbles_won",
        "Fouled":   "fouls_drawn_raw",     # không dùng, chỉ lưu tạm
    },
    "Defensive": {
        "TotalTackles":  "tackles",
        "Interceptions": "interceptions",
        "Clearances":    "clearances",
        "Fouls":         "fouls_committed",
    },
    "Passing": {
        "Passes": "passes_total",
        # Bỏ PA% vì đã có từ Summary
    },
}

# Cột output theo field_analysis.md (bỏ các trường không cần)
OUTPUT_COLS = [
    "match_id", "match_date", "season",
    "home_team", "away_team", "home_score", "away_score",
    "player_id", "name", "team", "is_home", "position", "age",
    "is_first_eleven", "is_man_of_match", "minutes_played",
    "rating", "goals", "assists",
    "shots_total", "shots_on_target", "key_passes",
    "passes_completed", "passes_total", "pass_accuracy",
    "tackles", "interceptions", "clearances",
    "aerial_won", "aerial_lost",
    "dribbles_won", "dribbles_attempted",
    "fouls_committed", "saves",
]

DELAY_PAGE    = 4
DELAY_TAB     = 2
DELAY_BETWEEN = 2


# ─────────────────────────────────────────────────────────────────
# DRIVER
# ─────────────────────────────────────────────────────────────────
def build_driver(headless=True):
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--remote-debugging-port=9222")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-renderer-backgrounding")
    options.add_argument("--disable-background-timer-throttling")
    options.add_argument("--single-process")
    options.add_argument("--memory-pressure-off")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.binary_location = "/usr/bin/google-chrome"
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


# ─────────────────────────────────────────────────────────────────
# CALENDAR NAVIGATION
# ─────────────────────────────────────────────────────────────────
def get_current_month(driver) -> str:
    try:
        el = driver.find_element(By.CSS_SELECTOR, SEL_MONTH_LABEL)
        return el.text.strip().split("\n")[0].strip()
    except Exception:
        return ""


def click_prev_month(driver) -> bool:
    try:
        btns = driver.find_elements(By.CSS_SELECTOR, SEL_PREV_BTN)
        if not btns:
            return False
        try:
            btns[0].click()
        except Exception:
            driver.execute_script("arguments[0].click();", btns[0])
        time.sleep(3)
        return True
    except Exception:
        return False


def navigate_to_month(driver, target: str) -> bool:
    for attempt in range(12):
        current = get_current_month(driver)
        print(f"    [{attempt+1}] '{current}' → '{target}'")
        if current == target:
            return True
        cur_ord = MONTH_ORDER.get(current, -1)
        tgt_ord = MONTH_ORDER.get(target, -1)
        if cur_ord == -1 or cur_ord <= tgt_ord:
            return False
        if not click_prev_month(driver):
            return False
    return False


# ─────────────────────────────────────────────────────────────────
# FIXTURES PAGE
# ─────────────────────────────────────────────────────────────────
def scroll_full_page(driver):
    last_h = 0
    for _ in range(15):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.6)
        new_h = driver.execute_script("return document.body.scrollHeight")
        if new_h == last_h:
            break
        last_h = new_h
    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(1)


def get_match_info_on_page(driver) -> list[dict]:
    scroll_full_page(driver)

    score_links = driver.find_elements(By.CSS_SELECTOR, SEL_SCORE)
    stats_links = driver.find_elements(By.CSS_SELECTOR, SEL_STATS_BTN)

    stats_href_map = {}
    for btn in stats_links:
        href = btn.get_attribute("href") or ""
        m = re.search(r"/matches/(\d+)/", href)
        if m:
            stats_href_map[m.group(1)] = href

    print(f"    score_links={len(score_links)} | stats_links={len(stats_links)}")

    matches, seen = [], set()
    for i, sl in enumerate(score_links):
        try:
            driver.execute_script("arguments[0].scrollIntoView(true);", sl)
            time.sleep(0.3)

            href = sl.get_attribute("href") or ""
            m    = re.search(r"/matches/(\d+)/", href)
            if not m:
                continue
            match_id = m.group(1)
            if match_id in seen:
                continue
            seen.add(match_id)

            score_raw  = sl.text.strip().replace("\n", ":")
            parts      = score_raw.split(":")
            home_score = parts[0].strip() if parts else ""
            away_score = parts[1].strip() if len(parts) > 1 else ""

            try:
                container = sl.find_element(By.XPATH,
                    "ancestor::div[.//a[contains(@class,'teamNameText')]][1]"
                )
                team_els  = container.find_elements(By.CSS_SELECTOR, SEL_TEAM_NAME)
                home_team = team_els[0].text.strip() if len(team_els) > 0 else ""
                away_team = team_els[1].text.strip() if len(team_els) > 1 else ""
            except Exception:
                all_teams = driver.find_elements(By.CSS_SELECTOR, SEL_TEAM_NAME)
                home_team = all_teams[i*2].text.strip()   if i*2   < len(all_teams) else ""
                away_team = all_teams[i*2+1].text.strip() if i*2+1 < len(all_teams) else ""

            stats_url = re.sub(r"/live/", "/livestatistics/",
                               stats_href_map.get(match_id, ""))

            matches.append({
                "match_id": match_id, "home_team": home_team,
                "away_team": away_team, "home_score": home_score,
                "away_score": away_score, "stats_url": stats_url,
            })
            print(f"    [{i:>2}] [{match_id}] {home_team:20} {score_raw:>5}  {away_team}")

        except Exception as e:
            print(f"    [ERR] match {i}: {e}")

    return matches


# ─────────────────────────────────────────────────────────────────
# MATCH PAGE HELPERS
# ─────────────────────────────────────────────────────────────────
def get_match_date(driver) -> str:
    src = driver.page_source
    for pat in [
        r'Date:\s*</[^>]+>\s*([^<]+)',
        r'"matchDate"\s*:\s*"([^"]+)"',
        r'((?:Mon|Tue|Wed|Thu|Fri|Sat|Sun),\s+\d{2}-\w+-\d{2})',
        r'(\d{4}-\d{2}-\d{2})',
    ]:
        m = re.search(pat, src)
        if m:
            return m.group(1).strip()
    return ""


def get_page_headers(driver) -> list[str]:
    """Lấy headers từ table đầu tiên có header trên trang."""
    for tbl in driver.find_elements(By.CSS_SELECTOR, "table.grid"):
        headers = [h.text.strip() for h in
                   tbl.find_elements(By.CSS_SELECTOR, "thead th, thead td")]
        headers = [h for h in headers if h]
        if headers:
            return headers
    return []


def scroll_and_get_tables(driver) -> list:
    """
    Scroll toàn trang để trigger lazy render,
    sau đó scroll từng table vào viewport rồi mới check rows có text.
    """
    # Scroll toàn trang 2 lần
    for _ in range(2):
        for _ in range(10):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(0.6)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)

    valid = []
    for tbl in driver.find_elements(By.CSS_SELECTOR, "table.grid"):
        # Scroll table vào viewport TRƯỚC
        driver.execute_script("arguments[0].scrollIntoView(true);", tbl)
        time.sleep(0.5)

        # Scroll row đầu vào viewport
        trs = tbl.find_elements(By.CSS_SELECTOR, "tbody tr")
        if not trs:
            continue
        driver.execute_script("arguments[0].scrollIntoView(true);", trs[0])
        time.sleep(0.3)

        # Check text SAU KHI đã scroll vào viewport
        rows_with_text = sum(1 for tr in trs[:5] if tr.text.strip())
        if rows_with_text >= 3:
            valid.append(tbl)

    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(1)
    return valid


def parse_table_obj(driver, table, fallback_headers: list) -> list[dict]:
    """
    Dùng RAW headers (kể cả rỗng '') để map đúng index với cell.
    fallback_headers dùng khi table không có thead (away table lazy).
    """
    # Lấy raw headers GIỮ NGUYÊN header rỗng
    raw_headers = [h.text.strip() for h in
                   table.find_elements(By.CSS_SELECTOR, "thead th, thead td")]

    # Nếu không có headers → dùng fallback
    if not any(raw_headers):
        raw_headers = fallback_headers

    if not raw_headers:
        return []

    rows_data = []
    for tr in table.find_elements(By.CSS_SELECTOR, "tbody tr"):
        driver.execute_script("arguments[0].scrollIntoView(true);", tr)
        time.sleep(0.1)
        cells = tr.find_elements(By.TAG_NAME, "td")
        if not cells:
            continue
        row = {}
        for j, cell in enumerate(cells):
            if j >= len(raw_headers):
                break
            header = raw_headers[j]
            if header:  # bỏ qua header rỗng
                row[header] = cell.text.strip()
        rows_data.append(row)
    return rows_data


def parse_key_events(html: str) -> dict:
    """
    Parse Key Events HTML dùng data attributes chính xác:
    data-type="16" + goalnormal  → goal
    data-type="16" + owngoal     → own goal (không tính)
    data-type="1"  + assist      → assist
    data-type="17" + yellowcard  → yellow card
    data-type="9"  + redcard     → red card
    """
    events = {"goals": 0, "assists": 0, "yellow_cards": 0, "red_cards": 0}

    # Tìm tất cả incident spans
    icons = re.findall(r'<span[^>]+data-type="(\d+)"[^>]*/>', html) or \
            re.findall(r'<span[^>]+data-type="(\d+)"[^>]*></span>', html)

    # Parse từng icon span đầy đủ
    spans = re.findall(r'<span[^>]+data-type="[^"]*"[^>]*>', html)
    for span in spans:
        dtype = re.search(r'data-type="(\d+)"', span)
        if not dtype:
            continue
        t = dtype.group(1)

        if t == "16":  # Shot event
            if "goalnormal" in span or "goalopen" in span or "goalcorner" in span:
                if "owngoal" not in span:
                    events["goals"] += 1
        elif t == "1":  # Pass event
            if "assist" in span and "intentionalassist" not in span:
                events["assists"] += 1
            elif "intentionalassist" in span:
                events["assists"] += 1
        elif t == "17":  # Card
            if "yellowcard" in span:
                events["yellow_cards"] += 1
            elif "redcard" in span or "yellowredcard" in span:
                events["red_cards"] += 1
        elif t == "9":  # Red card trực tiếp
            events["red_cards"] += 1

    return events


def get_player_ids(driver, table) -> dict:
    """{ player_name: player_id } từ links /players/ trong table."""
    pids = {}
    for tr in table.find_elements(By.CSS_SELECTOR, "tbody tr"):
        driver.execute_script("arguments[0].scrollIntoView(true);", tr)
        time.sleep(0.1)
        for lnk in tr.find_elements(By.CSS_SELECTOR, "a[href*='/players/']"):
            href = lnk.get_attribute("href") or ""
            m    = re.search(r"/players/(\d+)/", href)
            txt  = lnk.text.strip()
            if m and "\n" in txt:
                name = txt.split("\n")[-1].strip()
                if name:
                    pids[name] = m.group(1)
                    break
    return pids


def click_tab(driver, tab_name: str) -> bool:
    for li in driver.find_elements(By.CSS_SELECTOR, "ul.tabs.border-bottom li"):
        if li.text.strip() == tab_name:
            anchors = li.find_elements(By.TAG_NAME, "a")
            driver.execute_script("arguments[0].click();", anchors[0] if anchors else li)
            time.sleep(DELAY_TAB)
            return True
    try:
        el = driver.find_element(By.XPATH,
            f"//ul[contains(@class,'tabs')]"
            f"//*[normalize-space(text())='{tab_name}']"
        )
        driver.execute_script("arguments[0].click();", el)
        time.sleep(DELAY_TAB)
        return True
    except Exception:
        return False


# ─────────────────────────────────────────────────────────────────
# PARSE PLAYER CELL
# ─────────────────────────────────────────────────────────────────
def parse_player_cell(raw: str) -> dict:
    """
    "1\\nKarl Darlow\\n35, GK"               → first eleven, 90 mins
    "12\\nSoungoutou Magassa\\n22, Sub (84′)" → sub on 84', 6 mins
    "16\\nAaron Wan-Bissaka\\n28, Sub"        → unused sub, 0 mins
    """
    lines = [l.strip() for l in raw.strip().split("\n") if l.strip()]
    if not lines:
        return {}

    idx      = 1 if lines[0].isdigit() else 0
    name     = lines[idx]   if idx < len(lines) else ""
    info     = lines[idx+1] if idx+1 < len(lines) else ""

    age, position           = "", ""
    is_first_eleven         = True
    minutes_played          = 90

    if info:
        age_m = re.match(r"(\d+)", info)
        age   = age_m.group(1) if age_m else ""

        pos_m = re.search(
            r",\s*(GK|DC|DR|DL|DMC|DMR|DML|MC|MR|ML|AMC|AMR|AML|FW|FWR|FWL|Sub)\b",
            info
        )
        position = pos_m.group(1) if pos_m else ""
        min_m    = re.search(r"\((\d+)[′']\)", info)

        if position == "Sub":
            is_first_eleven = False
            minutes_played  = (90 - int(min_m.group(1))) if min_m else 0
        elif min_m:
            minutes_played  = int(min_m.group(1))

    return {
        "name": name, "age": age, "position": position,
        "is_first_eleven": 1 if is_first_eleven else 0,
        "minutes_played":  minutes_played,
    }


# ─────────────────────────────────────────────────────────────────
# CRAWL ONE MATCH
# ─────────────────────────────────────────────────────────────────
def init_player(pinfo: dict) -> dict:
    return {
        "name":            pinfo.get("name", ""),
        "age":             pinfo.get("age", ""),
        "position":        pinfo.get("position", ""),
        "is_first_eleven": pinfo.get("is_first_eleven", 0),
        "minutes_played":  pinfo.get("minutes_played", 0),
        "rating": "",
        "goals": 0, "assists": 0, "saves": 0,
        "shots_total": 0, "shots_on_target": 0, "key_passes": 0,
        "passes_completed": 0, "passes_total": 0, "pass_accuracy": 0,
        "tackles": 0, "interceptions": 0, "clearances": 0,
        "aerial_won": 0, "aerial_lost": 0,
        "dribbles_won": 0, "dribbles_attempted": 0,
        "fouls_committed": 0,
    }


def crawl_one_match(driver, match_info: dict, month: str) -> list[dict]:
    match_id   = match_info["match_id"]
    home_team  = match_info["home_team"]
    away_team  = match_info["away_team"]
    home_score = match_info["home_score"]
    away_score = match_info["away_score"]
    match_date = get_match_date(driver)

    for el in driver.find_elements(By.TAG_NAME, "a"):
        if "Player Statistics" in el.text:
            driver.execute_script("arguments[0].click();", el)
            time.sleep(2)
            break

    data = {"home": {}, "away": {}}
    pids = {"home": {}, "away": {}}

    for tab_name, field_map in TAB_FIELD_MAP.items():
        if not click_tab(driver, tab_name):
            print(f"      [WARN] Không click tab {tab_name}")
            continue

        valid = scroll_and_get_tables(driver)
        if len(valid) < 2:
            print(f"      [WARN] {tab_name}: chỉ có {len(valid)} table")
            continue

        home_tbl, away_tbl = valid[0], valid[1]
        fallback_headers = [h.text.strip() for h in
                            home_tbl.find_elements(By.CSS_SELECTOR, "thead th, thead td")]
        print(f"      {tab_name}: home=✓ away=✓ | headers={[h for h in fallback_headers if h][:5]}")

        for side, tbl, team in [("home", home_tbl, home_team),
                                 ("away", away_tbl, away_team)]:
            if tab_name == "Summary":
                pids[side] = get_player_ids(driver, tbl)

            rows = parse_table_obj(driver, tbl, fallback_headers)
            print(f"        {side}: {len(rows)} rows parsed")  # debug

            for row in rows:                                    # ← indent đúng
                pinfo = parse_player_cell(row.get("Player", ""))
                name  = pinfo.get("name", "")
                if not name:
                    continue
                if name not in data[side]:
                    data[side][name] = init_player(pinfo)
                for ws_col, field in field_map.items():        # ← nằm TRONG for row
                    val = row.get(ws_col, "")
                    if val and val not in ("", "-"):
                        if data[side][name].get(field) in (0, "", None) or field == "rating":
                            data[side][name][field] = val

    print(f"      data: home={len(data['home'])} players, away={len(data['away'])} players")  # debug

    # Key Events
    click_tab(driver, "Summary")
    valid = scroll_and_get_tables(driver)
    if len(valid) >= 2:
        fallback = [h.text.strip() for h in
                    valid[0].find_elements(By.CSS_SELECTOR, "thead th, thead td")]
        for side, tbl in [("home", valid[0]), ("away", valid[1])]:
            for tr in tbl.find_elements(By.CSS_SELECTOR, "tbody tr"):
                driver.execute_script("arguments[0].scrollIntoView(true);", tr)
                time.sleep(0.1)
                cells = tr.find_elements(By.TAG_NAME, "td")
                if not cells:
                    continue
                name = parse_player_cell(cells[0].text.strip()).get("name", "")
                if not name or name not in data[side]:
                    continue
                html = cells[-1].get_attribute("innerHTML") or ""
                if not html.strip():
                    continue
                events = parse_key_events(html)
                for field, val in events.items():
                    if val > 0:
                        data[side][name][field] = val

    # Build records
    records = []
    for side, team_name in [("home", home_team), ("away", away_team)]:
        for name, s in data[side].items():
            print(f"        Check {name}: minutes={s.get('minutes_played')} rating={s.get('rating')!r}")  # debug
            if s.get("minutes_played", 0) == 0:
                continue
            if not s.get("rating") or str(s.get("rating")) in ("", "-"):
                continue
            records.append({
                "match_id": match_id, "match_date": match_date,
                "season": "2025/2026",
                "home_team": home_team, "away_team": away_team,
                "home_score": home_score, "away_score": away_score,
                "player_id": pids[side].get(name, ""),
                "name": name, "team": team_name,
                "is_home": 1 if side == "home" else 0,
                "position": s.get("position", ""),
                "age": s.get("age", ""),
                "is_first_eleven": s.get("is_first_eleven", 0),
                "is_man_of_match": 0,
                "minutes_played": s.get("minutes_played", 0),
                "rating": s.get("rating", ""),
                "goals": s.get("goals", 0),
                "assists": s.get("assists", 0),
                "shots_total": s.get("shots_total", 0),
                "shots_on_target": s.get("shots_on_target", 0),
                "key_passes": s.get("key_passes", 0),
                "passes_completed": s.get("passes_completed", 0),
                "passes_total": s.get("passes_total", 0),
                "pass_accuracy": s.get("pass_accuracy", 0),
                "tackles": s.get("tackles", 0),
                "interceptions": s.get("interceptions", 0),
                "clearances": s.get("clearances", 0),
                "aerial_won": s.get("aerial_won", 0),
                "aerial_lost": s.get("aerial_lost", 0),
                "dribbles_won": s.get("dribbles_won", 0),
                "dribbles_attempted": s.get("dribbles_attempted", 0),
                "fouls_committed": s.get("fouls_committed", 0),
                "saves": s.get("saves", 0),
            })
     # Cuối hàm crawl_one_match, sau khi build records xong, thêm:
    if records:
       max_rating = max(
        float(r["rating"]) for r in records
        if r.get("rating") and str(r["rating"]) not in ("", "-")
    )
       motm_set = False
       for r in records:
            try:
                if float(r["rating"]) == max_rating and not motm_set:
                    r["is_man_of_match"] = 1
                    motm_set = True
            except (ValueError, TypeError):
                pass
    return records



# ─────────────────────────────────────────────────────────────────
# CRAWL ONE MONTH
# ─────────────────────────────────────────────────────────────────
def crawl_month(driver, month_label: str) -> list[dict]:
    all_records = []
    matches     = get_match_info_on_page(driver)
    print(f"  Tổng {len(matches)} trận")

    for i, match_info in enumerate(matches):
        stats_url = match_info.get("stats_url", "")
        if not stats_url:
            continue

        print(f"\n  [{i+1}/{len(matches)}] [{match_info['match_id']}] "
              f"{match_info['home_team']} "
              f"{match_info['home_score']}:{match_info['away_score']} "
              f"{match_info['away_team']}")

        driver.get(stats_url)
        time.sleep(DELAY_PAGE)

        records = crawl_one_match(driver, match_info, month_label)
        print(f"    ✓ {len(records)} players")
        all_records.extend(records)

        # Quay về fixtures và navigate lại đúng tháng
        driver.get(FIXTURES_URL)
        time.sleep(DELAY_PAGE)
        if get_current_month(driver) != month_label:
            navigate_to_month(driver, month_label)
            time.sleep(1)
        scroll_full_page(driver)
        time.sleep(DELAY_BETWEEN)

    return all_records


# ─────────────────────────────────────────────────────────────────
# TEST: 1 TRẬN
# ─────────────────────────────────────────────────────────────────
def main_test():
    print("[TEST] Crawl thử 1 trận...")
    driver = build_driver(headless=True)
    try:
        driver.get(FIXTURES_URL)
        time.sleep(6)
        print(f"Tháng hiện tại: '{get_current_month(driver)}'")

        matches = get_match_info_on_page(driver)
        if not matches:
            print("[ERR] Không có trận")
            return

        test_match = matches[0]
        print(f"\nCrawl: [{test_match['match_id']}] "
              f"{test_match['home_team']} "
              f"{test_match['home_score']}:{test_match['away_score']} "
              f"{test_match['away_team']}")

        driver.get(test_match["stats_url"])
        time.sleep(DELAY_PAGE)

        records = crawl_one_match(driver, test_match, "test")
        print(f"\n[RESULT] {len(records)} players")
        if not records:
            return

        df = pd.DataFrame(records)
        print(f"\n{'='*55}\nKIỂM TRA TRƯỜNG:\n{'='*55}")
        for col in OUTPUT_COLS:
            if col not in df.columns:
                print(f"  {col:25} | ❌ THIẾU CỘT")
                continue
            non_default = df[col].replace("", pd.NA).dropna()
            non_default = non_default[~non_default.astype(str).isin(["0", "0.0"])]
            status = "✓" if len(non_default) > 0 else "⚠ toàn 0/rỗng"
            print(f"  {col:25} | {len(non_default):>3}/{len(df)} | {status}")

        print(f"\n{'='*55}\nPREVIEW 10 PLAYERS:\n{'='*55}")
        print(df[["name","team","position","minutes_played","rating",
                  "goals","assists","shots_total","tackles",
                  "interceptions","clearances","dribbles_won"]].head(10).to_string())

        df[OUTPUT_COLS].to_csv("test_result.csv", index=False, encoding="utf-8-sig")
        print(f"\n✓ Lưu → test_result.csv")

    finally:
        driver.quit()


# ─────────────────────────────────────────────────────────────────
# MAIN: TOÀN BỘ MÙA GIẢI
# ─────────────────────────────────────────────────────────────────
def main():
    all_records, failed = [], []

    driver_tmp = build_driver(headless=True)
    try:
        driver_tmp.get(FIXTURES_URL)
        time.sleep(5)
        current_month = get_current_month(driver_tmp)
        print(f"[INFO] Tháng hiện tại: '{current_month}'")
    finally:
        driver_tmp.quit()

    months_to_crawl = [
        m for m in TARGET_MONTHS
        if MONTH_ORDER.get(m, -1) <= MONTH_ORDER.get(current_month, 99)
    ]
    print(f"[INFO] Crawl {len(months_to_crawl)} tháng: {months_to_crawl}")

    for i, target_month in enumerate(months_to_crawl):
        print(f"\n{'='*60}\n[{i+1}/{len(months_to_crawl)}] {target_month}\n{'='*60}")

        driver = build_driver(headless=True)
        try:
            driver.get(FIXTURES_URL)
            time.sleep(6)

            if not navigate_to_month(driver, target_month):
                print(f"  [SKIP] Không navigate được")
                failed.append(target_month)
                continue
            time.sleep(2)

            if not driver.find_elements(By.CSS_SELECTOR, SEL_SCORE):
                print(f"  [SKIP] Chưa có trận")
                continue

            records = crawl_month(driver, target_month)
            all_records.extend(records)
            print(f"\n  ✓ {target_month}: {len(records)} records")

        except Exception as e:
            print(f"  [ERR] {target_month}: {e}")
            failed.append(target_month)
        finally:
            driver.quit()
            time.sleep(3)

        if all_records:
            df_tmp = pd.DataFrame(all_records)
            cols   = [c for c in OUTPUT_COLS if c in df_tmp.columns]
            df_tmp[cols].to_csv("motm_data_temp.csv", index=False, encoding="utf-8-sig")
            print(f"  [SAVE] {len(all_records)} records → motm_data_temp.csv")

    if not all_records:
        print("[ERR] Không có dữ liệu!")
        return

    df   = pd.DataFrame(all_records)
    cols = [c for c in OUTPUT_COLS if c in df.columns]
    df[cols].to_csv("motm_data_2025_2026.csv", index=False, encoding="utf-8-sig")

    print(f"\n{'='*60}")
    print(f"✓ HOÀN TẤT: {len(df)} records | {df['match_id'].nunique()} matches")
    print(f"  Tháng lỗi : {failed if failed else 'Không có'}")
    print(f"  Output    : motm_data_2025_2026.csv")
    print(f"{'='*60}")
    print(df[["match_id","match_date","home_team","away_team",
              "name","position","minutes_played","rating",
              "goals","assists","tackles"]].head(10).to_string())


# ─────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()  # bỏ comment khi sẵn sàng crawl toàn bộ

In [2]:
# ================================================================
# MOTM Predictor - Data Cleaning
# Google Colab Notebook
# ================================================================
# Upload file PlayerCrawl.xlsx lên Colab trước khi chạy

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────────────────────────────────────────
# CELL 1: Load dữ liệu
# ────────────────────────────────────────────────────────────────
df24 = pd.read_excel("PlayerCrawl.xlsx", sheet_name="20242025")
df25 = pd.read_excel("PlayerCrawl.xlsx", sheet_name="20252026")

print(f"2024/2025: {df24.shape}")
print(f"2025/2026: {df25.shape}")

# ────────────────────────────────────────────────────────────────
# CELL 2: Fix match_date
# Vấn đề: 2024/2025 dùng "2025-02-15", 2025/2026 dùng "Fri, 01-May-26"
# ────────────────────────────────────────────────────────────────
def parse_match_date(val):
    """Parse nhiều format ngày khác nhau về chuẩn YYYY-MM-DD."""
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip()
    for fmt in [
        "%Y-%m-%d",           # 2025-02-15
        "%a, %d-%b-%y",       # Fri, 01-May-26
        "%a, %d-%b-%Y",       # Fri, 01-May-2026
        "%d/%m/%Y",
        "%d-%m-%Y",
    ]:
        try:
            return pd.to_datetime(s, format=fmt)
        except Exception:
            pass
    return pd.NaT

df24["match_date"] = df24["match_date"].apply(parse_match_date)
df25["match_date"] = df25["match_date"].apply(parse_match_date)

print("match_date 2024/2025 NaT:", df24["match_date"].isna().sum())
print("match_date 2025/2026 NaT:", df25["match_date"].isna().sum())
print("Sample 2025/2026:", df25["match_date"].head(3).tolist())

# ────────────────────────────────────────────────────────────────
# CELL 3: Fix rating
# Vấn đề: 2025/2026 rating bị lẫn datetime object (Excel đọc sai)
# VD: "6.43", datetime(2026,12,7) thay vì "6.07"
# ────────────────────────────────────────────────────────────────
def fix_rating(val):
    """
    Chuyển rating về float.
    Nếu là datetime → trích xuất phần thập phân từ ngày (day.month → 7.12 → 6.07? không đáng tin)
    → đánh dấu NaN để xử lý sau.
    """
    if pd.isna(val):
        return np.nan
    if isinstance(val, (pd.Timestamp, type(pd.NaT))):
        # datetime bị đọc nhầm → không thể recover → NaN
        return np.nan
    try:
        f = float(str(val).strip())
        if 0 < f < 11:   # rating hợp lệ từ 0-10
            return f
        return np.nan
    except Exception:
        return np.nan

df25["rating"] = df25["rating"].apply(fix_rating)

print(f"\nRating NaN trong 2025/2026: {df25['rating'].isna().sum()} / {len(df25)}")
print(f"Rating range: {df25['rating'].min():.2f} – {df25['rating'].max():.2f}")

# ────────────────────────────────────────────────────────────────
# CELL 4: Fix pass_accuracy
# Vấn đề: 2025/2026 có mix string "68.3" và int 92
# ────────────────────────────────────────────────────────────────
def fix_float(val):
    if pd.isna(val):
        return np.nan
    try:
        f = float(str(val).strip())
        return f if f >= 0 else np.nan
    except Exception:
        return np.nan

df25["pass_accuracy"] = df25["pass_accuracy"].apply(fix_float)
df24["pass_accuracy"] = df24["pass_accuracy"].apply(fix_float)

# ────────────────────────────────────────────────────────────────
# CELL 5: Fix minutes_played
# Vấn đề: 2025/2026 có giá trị âm (min = -9) do parse sai Sub time
# ────────────────────────────────────────────────────────────────
print(f"\nminutes_played < 0 (2025/2026): {(df25['minutes_played'] < 0).sum()}")
print(df25[df25['minutes_played'] < 0][['name','position','minutes_played']].head())

# Fix: clip về 0
df25["minutes_played"] = df25["minutes_played"].clip(lower=0)
df24["minutes_played"] = df24["minutes_played"].clip(lower=0)

print(f"Sau fix - minutes_played range 2025/2026: {df25['minutes_played'].min()} – {df25['minutes_played'].max()}")

# ────────────────────────────────────────────────────────────────
# CELL 6: Merge 2 mùa
# ────────────────────────────────────────────────────────────────
df = pd.concat([df24, df25], ignore_index=True)
print(f"\nSau merge: {df.shape}")
print(f"Seasons: {df['season'].unique()}")

# ────────────────────────────────────────────────────────────────
# CELL 7: Đảm bảo kiểu dữ liệu đúng
# ────────────────────────────────────────────────────────────────
# Numeric cols
num_cols = [
    "home_score", "away_score", "is_home", "is_first_eleven",
    "is_man_of_match", "age", "minutes_played",
    "rating", "goals", "assists",
    "shots_total", "shots_on_target", "key_passes",
    "passes_completed", "passes_total", "pass_accuracy",
    "tackles", "interceptions", "clearances",
    "aerial_won", "dribbles_won", "fouls_committed",
]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# String cols
str_cols = ["season", "home_team", "away_team", "name", "team", "position"]
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

print("Dtypes sau fix:")
print(df[num_cols].dtypes)

# ────────────────────────────────────────────────────────────────
# CELL 8: Loại bỏ rows không hợp lệ
# ────────────────────────────────────────────────────────────────
before = len(df)

# 1. Loại cầu thủ không ra sân
df = df[df["minutes_played"] > 0]
print(f"Sau loại minutes_played=0: {len(df)} (bỏ {before - len(df)} rows)")

# 2. Loại rows không có rating
before2 = len(df)
df = df[df["rating"].notna() & (df["rating"] > 0)]
print(f"Sau loại rating=NaN/0: {len(df)} (bỏ {before2 - len(df)} rows)")

# 3. Loại duplicate (cùng match_id + player_id)
before3 = len(df)
df = df.drop_duplicates(subset=["match_id", "player_id"], keep="first")
print(f"Sau loại duplicate: {len(df)} (bỏ {before3 - len(df)} rows)")

# ────────────────────────────────────────────────────────────────
# CELL 9: Fill missing numeric với 0
# (shots, tackles... = 0 là hợp lệ, không phải missing thực sự)
# ────────────────────────────────────────────────────────────────
fill_zero_cols = [
    "goals", "assists", "shots_total", "shots_on_target", "key_passes",
    "passes_completed", "passes_total", "tackles", "interceptions",
    "clearances", "aerial_won", "dribbles_won", "fouls_committed",
]
df[fill_zero_cols] = df[fill_zero_cols].fillna(0)

# pass_accuracy: fill bằng median theo position
df["pass_accuracy"] = df.groupby("position")["pass_accuracy"].transform(
    lambda x: x.fillna(x.median())
)

print(f"\nNull count sau fill:\n{df[num_cols].isnull().sum()[df[num_cols].isnull().sum() > 0]}")

# ────────────────────────────────────────────────────────────────
# CELL 10: Position grouping & encoding
# ────────────────────────────────────────────────────────────────
POSITION_GROUP = {
    "GK":  "GK",
    "DC":  "DEF", "DR": "DEF", "DL": "DEF",
    "DMC": "MID", "DMR": "MID", "DML": "MID",
    "MC":  "MID", "MR": "MID",  "ML": "MID",
    "AMC": "ATT", "AMR": "ATT", "AML": "ATT",
    "FW":  "ATT", "FWR": "ATT", "FWL": "ATT",
    "Sub": "Sub",
}
POSITION_ENCODE = {"GK": 0, "DEF": 1, "MID": 2, "ATT": 3, "Sub": 4}

df["position_group"]   = df["position"].map(POSITION_GROUP).fillna("MID")
df["position_encoded"] = df["position_group"].map(POSITION_ENCODE)

print("\nPosition distribution:")
print(df["position_group"].value_counts())

# ────────────────────────────────────────────────────────────────
# CELL 11: Derived features
# ────────────────────────────────────────────────────────────────
# Score margin từ góc nhìn cầu thủ
df["score_margin"] = np.where(
    df["is_home"] == 1,
    df["home_score"] - df["away_score"],
    df["away_score"] - df["home_score"],
)

# Goal involvement
df["goal_involvement"] = df["goals"] + df["assists"]

# Aerial win rate (tránh chia 0)
# Lưu ý: aerial_lost không có trong data → dùng aerial_won trực tiếp
df["aerial_won"] = df["aerial_won"].fillna(0)

# Shot accuracy (tránh chia 0)
df["shot_accuracy"] = np.where(
    df["shots_total"] > 0,
    df["shots_on_target"] / df["shots_total"],
    0.0,
)

# Minutes ratio (phút thực tế / 90)
df["minutes_ratio"] = (df["minutes_played"] / 90).clip(0, 1.3)

print("\nDerived features sample:")
print(df[["name","score_margin","goal_involvement","shot_accuracy","minutes_ratio"]].head(5))

# ────────────────────────────────────────────────────────────────
# CELL 12: Rolling features (form 5 trận gần nhất)
# Sort theo player_id + match_date để tính đúng thứ tự
# ────────────────────────────────────────────────────────────────
df = df.sort_values(["player_id", "match_date"]).reset_index(drop=True)

def rolling_shift(series, window=5):
    """Rolling mean của 5 trận TRƯỚC (shift 1 để tránh data leakage)."""
    return series.shift(1).rolling(window, min_periods=1).mean()

grp = df.groupby("player_id")

df["rolling_rating_5"]      = grp["rating"].transform(rolling_shift)
df["rolling_goals_5"]       = grp["goals"].transform(rolling_shift)
df["rolling_assists_5"]     = grp["assists"].transform(rolling_shift)
df["rolling_shots_5"]       = grp["shots_total"].transform(rolling_shift)
df["rolling_key_passes_5"]  = grp["key_passes"].transform(rolling_shift)
df["rolling_tackles_5"]     = grp["tackles"].transform(rolling_shift)

# Fill NaN rolling (trận đầu tiên chưa có lịch sử) bằng giá trị hiện tại
rolling_cols = [c for c in df.columns if c.startswith("rolling_")]
for col in rolling_cols:
    base_col = col.replace("rolling_", "").replace("_5", "")
    if base_col in df.columns:
        df[col] = df[col].fillna(df[base_col])

print(f"\nRolling features: {rolling_cols}")
print(df[["name","match_date","rating","rolling_rating_5"]].head(8))

# ────────────────────────────────────────────────────────────────
# CELL 13: Fix is_man_of_match
# Đảm bảo mỗi trận chỉ có đúng 1 MOTM = cầu thủ rating cao nhất
# ────────────────────────────────────────────────────────────────
# Reset về 0 trước
df["is_man_of_match"] = 0

for match_id, group in df.groupby("match_id"):
    valid = group[group["rating"].notna() & (group["rating"] > 0)]
    if valid.empty:
        continue
    max_idx = valid["rating"].idxmax()
    df.at[max_idx, "is_man_of_match"] = 1

motm_count = df["is_man_of_match"].sum()
match_count = df["match_id"].nunique()
print(f"\nMOTM count: {motm_count} / {match_count} matches")
print(f"Ratio MOTM=1: {motm_count/len(df)*100:.1f}%")

# Kiểm tra mỗi trận có đúng 1 MOTM không
motm_per_match = df.groupby("match_id")["is_man_of_match"].sum()
print(f"Trận có 0 MOTM: {(motm_per_match == 0).sum()}")
print(f"Trận có >1 MOTM: {(motm_per_match > 1).sum()}")

# ────────────────────────────────────────────────────────────────
# CELL 14: Final check
# ────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("FINAL DATA SUMMARY")
print("="*55)
print(f"Total rows       : {len(df):,}")
print(f"Unique matches   : {df['match_id'].nunique():,}")
print(f"Unique players   : {df['player_id'].nunique():,}")
print(f"Seasons          : {sorted(df['season'].unique())}")
print(f"Date range       : {df['match_date'].min().date()} → {df['match_date'].max().date()}")
print(f"MOTM = 1         : {df['is_man_of_match'].sum():,}")
print(f"\nNull count:")
null_summary = df.isnull().sum()
print(null_summary[null_summary > 0].to_string() if null_summary.any() else "  Không có null!")

print(f"\nRating distribution:")
print(df["rating"].describe())

print(f"\nPosition group:")
print(df["position_group"].value_counts())

# ────────────────────────────────────────────────────────────────
# CELL 15: Lưu file
# ────────────────────────────────────────────────────────────────
FINAL_COLS = [
    # Định danh
    "match_id", "match_date", "season",
    "home_team", "away_team", "home_score", "away_score",
    "player_id", "name", "team",
    # Trạng thái ra sân
    "is_home", "position", "position_group", "position_encoded",
    "age", "is_first_eleven", "minutes_played", "minutes_ratio",
    # Target
    "is_man_of_match",
    # Stats gốc
    "rating", "goals", "assists",
    "shots_total", "shots_on_target", "shot_accuracy", "key_passes",
    "passes_completed", "passes_total", "pass_accuracy",
    "tackles", "interceptions", "clearances",
    "aerial_won", "dribbles_won", "fouls_committed",
    # Context features
    "score_margin", "goal_involvement",
    # Rolling features
    "rolling_rating_5", "rolling_goals_5", "rolling_assists_5",
    "rolling_shots_5", "rolling_key_passes_5", "rolling_tackles_5",
]

df_final = df[[c for c in FINAL_COLS if c in df.columns]]
df_final.to_csv("motm_clean.csv", index=False, encoding="utf-8-sig")
print(f"\n✓ Lưu → motm_clean.csv | shape={df_final.shape}")
print(df_final.head(3).to_string())

2024/2025: (15188, 31)
2025/2026: (5003, 31)
match_date 2024/2025 NaT: 0
match_date 2025/2026 NaT: 0
Sample 2025/2026: [Timestamp('2026-05-01 00:00:00'), Timestamp('2026-05-01 00:00:00'), Timestamp('2026-05-01 00:00:00')]

Rating NaN trong 2025/2026: 791 / 5003
Rating range: 4.33 – 9.82

minutes_played < 0 (2025/2026): 78
                   name position  minutes_played
12      Wilfried Gnonto      Sub              -3
15         Daniel James      Sub              -3
103          Tom Edozie      Sub              -2
195           Leny Yoro      Sub              -5
283  Harrison Armstrong      Sub              -5
Sau fix - minutes_played range 2025/2026: 0 – 99

Sau merge: (20191, 31)
Seasons: ['2024/2025' '2025/2026']
Dtypes sau fix:
home_score            int64
away_score            int64
is_home               int64
is_first_eleven       int64
is_man_of_match       int64
age                   int64
minutes_played        int64
rating              float64
goals                 int64
assist